# ColorMNIST — Results

Tables only. Every number is read from `../ColorMNIST/dr_mmd_flow/results/*.csv` and
aggregated in the house order: average within a (data seed, repeat), then within a data seed,
then mean ± sd across the three seeds.

**Design.** 28×28 coloured digits, binary arm, confounding `w = 10`. Default cell:
`n_obs = 20000`, `M = 2000`, no-split nuisance and autoencoder, observational initialisation,
projected-Nystrom ridge, doubly robust target. The flow runs in the autoencoder latent space,
so pixel metrics are scored after decoding, and FID is the legacy TorchVision scorer matching
the DecFM convention (clean FID is recorded too, not reported). The `est_solver_*` stages re-run
the default design, so Sections 1–5 read the six main launcher stages only.

**How to read a ratio.** A flow row over the *count-matched* oracle floor, a second oracle draw
subsampled to the particle count; raw values do not compare across different `M`.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import analysis_tables as at

# The exact-ridge arm is invisible unless named: `load` selects on the `_projnys` filename
# token. `MAIN_TASKS` then drops the `est_solver_*` stages, which re-run the DEFAULT design and
# would otherwise pool into every table as extra repeats of cells the other stages cover.
WITH_EXACT = at.load("ColorMNIST", estimator=None, solvers=("projected_nystrom", "exact"))
ALL = WITH_EXACT[WITH_EXACT.solver == "projected_nystrom"]
MAIN_TASKS = ("est_headline", "est_nobs", "est_crossfit", "est_particles", "est_init",
              "est_wsweep")


def stage(frame, *names):
    """The rows of `frame` written by the named launcher stages."""
    return frame[frame.tag.astype(str).isin(names)]


MAIN = stage(ALL, *MAIN_TASKS)
DEFAULT = at.default_rows(MAIN, dataset="ColorMNIST", estimator=None)
DR = at.pin_estimator(DEFAULT)
SEEDS = sorted(int(s) for s in ALL.data_seed.unique())
print(f"{len(ALL)} rows   stages {sorted(ALL.tag.astype(str).unique())}")
print(f"seeds {SEEDS}   arms {sorted(int(a) for a in ALL.target_a.unique())}   "
      f"estimators {sorted(ALL.estimator.unique())}")
print(f"n_obs {sorted(int(n) for n in ALL.n_obs.unique())}   "
      f"M {sorted(int(m) for m in ALL.n_particles_requested.unique())}   "
      f"w {sorted(int(w) for w in ALL.w_conf.unique())}")
print(f"main grid {len(MAIN)} rows   default cell {len(DEFAULT)} rows = "
      f"{len(SEEDS)} seeds x 2 arms x 3 estimators")

432 rows   stages ['est_crossfit', 'est_headline', 'est_init', 'est_nobs', 'est_particles', 'est_solver_projected_nystrom', 'est_wsweep']
seeds [0, 1, 2]   arms [0, 1]   estimators ['dr', 'ipw', 'plugin']
n_obs [5000, 10000, 15000, 20000]   M [250, 500, 1000, 2000]   w [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
main grid 414 rows   default cell 18 rows = 3 seeds x 2 arms x 3 estimators


## 1. Headline

In [2]:
print(at.headline_table(MAIN, "ColorMNIST", at.DECFM_CMNIST))

ColorMNIST   (n_obs = 20000, M = 2000, seeds [0, 1, 2])

Method                SW2 ↓                FID ↓                Latent SW2 ↓         Color-W2 ↓
-----------------------------------------------------------------------------------------------
DecFM-Plugin          0.01137 ± 0.00078    32.005 ± 1.075       0.08206 ± 0.00754    0.01991 ± 0.00285
DecFM-DR-I            0.01392 ± 0.00090    33.650 ± 1.151       0.09217 ± 0.00657    0.03665 ± 0.01237
DecFM-DR-OT           0.01190 ± 0.00106    33.824 ± 1.219       0.07794 ± 0.00384    0.02840 ± 0.00866
IMMD-Flow (ours)      0.00930 ± 0.00030    30.644 ± 0.809       0.08004 ± 0.00114    0.02062 ± 0.00414

Ours: doubly robust target embedding, projected Nystrom, no-split nuisance, no-split autoencoder, P_Y initialisation.
FID is the legacy TorchVision Inception-v3 scorer, matching the reference convention.


Ours is the last row: SW2 0.00930 ± 0.00030 against 0.01137 ± 0.00078 for the strongest DecFM
baseline, and FID 30.644 ± 0.809 against 32.005–33.824. Latent SW2 (0.08004) and Color-W2
(0.02062) sit inside the range the three baselines span.

## 2. Reference rows

In [3]:
ROWS = [("No flow (logged arm)", "initial"), ("IMMD-Flow (ours)", "flow"),
        ("Autoencoder round-trip", "roundtrip"), ("Oracle floor", "floor")]
rows = [(label, [at.summarise(DR, f"{m}_{suffix}") if f"{m}_{suffix}" in DR.columns else None
                 for m in ("sw2", "fid_legacy", "latent_sw2", "colour_w2")])
        for label, suffix in ROWS]
print(at.plain_table(rows, ["SW2", "FID", "Latent SW2", "Color-W2"], first_column="Row",
                     label_width=24,
                     title=f"ColorMNIST   reference rows   (n_obs = 20000, M = 2000, "
                           f"seeds {SEEDS})",
                     note="Round-trip = the oracle sample pushed through the same "
                          "autoencoder, i.e. the decoder ceiling.\nLatent SW2 is measured "
                          "before decoding, so it has no round-trip row."))

ColorMNIST   reference rows   (n_obs = 20000, M = 2000, seeds [0, 1, 2])

Row                     SW2 ↓                FID ↓                Latent SW2 ↓         Color-W2 ↓
-------------------------------------------------------------------------------------------------
No flow (logged arm)    0.04341 ± 0.00026    43.071 ± 0.496       0.49675 ± 0.01573    0.25087 ± 0.00161
IMMD-Flow (ours)        0.00930 ± 0.00030    30.644 ± 0.809       0.08004 ± 0.00114    0.02062 ± 0.00414
Autoencoder round-trip  0.00813 ± 0.00072    30.620 ± 0.697       --                   0.01431 ± 0.00285
Oracle floor            0.00630 ± 0.00032    3.345 ± 0.103        0.07166 ± 0.00433    0.00674 ± 0.00099

Round-trip = the oracle sample pushed through the same autoencoder, i.e. the decoder ceiling.
Latent SW2 is measured before decoding, so it has no round-trip row.


The flow moves SW2 from 0.04341 on the logged arm to 0.00930, against an oracle floor of
0.00630. The FID column is the decoder, not the sampler: the round-trip ceiling is
30.620 ± 0.697 and the flow 30.644 ± 0.809, while the undecoded floor is 3.345. Latent SW2,
measured before decoding, is 0.08004 against a floor of 0.07166.

## 3. Target-embedding estimator

In [4]:
print(at.estimator_report(
    DEFAULT, "ColorMNIST",
    title=f"ColorMNIST   target embedding estimator   (n_obs = 20000, M = 2000, "
          f"seeds {SEEDS})"))

ColorMNIST   target embedding estimator   (n_obs = 20000, M = 2000, seeds [0, 1, 2])

Target embedding          SW2 ↓                FID ↓                Latent SW2 ↓         Color-W2 ↓
---------------------------------------------------------------------------------------------------
Plug-in (CME only)        0.00931 ± 0.00032    30.531 ± 0.843       0.08125 ± 0.00129    0.02166 ± 0.00428
IPW (ratios only)         0.00940 ± 0.00014    30.599 ± 1.293       0.09305 ± 0.01092    0.02375 ± 0.00339
Doubly robust (default)   0.00930 ± 0.00030    30.644 ± 0.809       0.08004 ± 0.00114    0.02062 ± 0.00414

One logged sample, one nuisance solve, one set of initial particles and one flow budget per run;
only the target weight vector differs.


ColorMNIST   the weight vector behind each row

Target embedding          sum v                ESS(v)               max |v|
---------------------------------------------------------------------------
Plug-in (CME only)        0.99634 ± 0.00249    1424 ± 

**Metrics.** The three tie on the decoded images: SW2 0.00931 / 0.00940 / 0.00930 and FID
30.531 / 30.599 / 30.644 for plug-in / IPW / DR, all inside one another's spreads. Latent SW2
separates them — IPW 0.09305 ± 0.01092 against 0.08125 and 0.08004 — and Color-W2 agrees.

**Weights.** Total mass is 0.99951 for DR against 0.99634 and 0.98905, closest to the 1 an
embedding of a probability measure carries. DR also carries the lowest signed ESS (1317 ± 125)
and the largest atom (0.006725); neither column is an objective.

## 4. Per arm

In [5]:
print(f"  {'arm':>3s} {'no flow':>9s} {'SW2':>9s} {'floor':>9s} {'ratio':>7s} "
      f"{'FID':>8s} {'latent SW2':>11s} {'latent floor':>13s}")
for arm in (0, 1):
    g = DR[DR.target_a == arm]
    logged, _ = at.summarise(g, "sw2_initial")
    flow, _ = at.summarise(g, "sw2_flow")
    floor, _ = at.summarise(g, "sw2_floor")
    fid, _ = at.summarise(g, "fid_legacy_flow")
    latent, _ = at.summarise(g, "latent_sw2_flow")
    latent_floor, _ = at.summarise(g, "latent_sw2_floor")
    print(f"  {arm:3d} {logged:9.5f} {flow:9.5f} {floor:9.5f} {flow / floor:6.2f}x "
          f"{fid:8.3f} {latent:11.5f} {latent_floor:13.5f}")

  arm   no flow       SW2     floor   ratio      FID  latent SW2  latent floor
    0   0.03853   0.00703   0.00542   1.30x   32.818     0.07213       0.06614
    1   0.04830   0.01158   0.00719   1.61x   28.469     0.08795       0.07719


Each arm intervenes to a different digit, so each has its own floor. Arm 0 flows to SW2 0.00703
against a floor of 0.00542 and arm 1 to 0.01158 against 0.00719, i.e. 1.30× and 1.61×; arm 0's
raw value is below arm 1's *floor*, so a raw cross-arm comparison inverts the ranking.

## 5. Ablations

One helper, five sweeps. Each changes one thing and leaves the rest at the default.

In [6]:
MAIN_DR = at.pin_estimator(MAIN).copy()
for metric in ("sw2", "latent_sw2"):
    MAIN_DR[metric + "_ratio"] = MAIN_DR[metric + "_flow"] / MAIN_DR[metric + "_floor_matched"]
FLOW = [("SW2", "sw2_flow"), ("SW2 / floor", "sw2_ratio"), ("FID", "fid_legacy_flow"),
        ("Latent SW2", "latent_sw2_flow"), ("Latent / floor", "latent_sw2_ratio")]


def sweep(frame, by, labels, title, first_column, order=None, metrics=FLOW,
          digits=(5, 2, 3, 5, 2), note=""):
    """One row per value of `by`, flow metrics beside their count-matched ratios."""
    print(at.ablation_table(frame, "ColorMNIST", by, labels, order=order, metrics=metrics,
                            arrow=False, digits=list(digits), first_column=first_column,
                            label_width=20, title=title, note=note))
    print()


NOSPLIT = MAIN_DR[MAIN_DR.dr_fit == "nosplit"]
sweep(stage(NOSPLIT, "est_nobs", "est_headline"), "n_obs", {}, first_column="n_obs",
      title="Observations   (M = 2000, no-split)")
sweep(stage(NOSPLIT, "est_particles", "est_headline"), "n_particles_requested", {},
      first_column="M (particles)", title="Particles   (n_obs = 20000, no-split)")
sweep(stage(MAIN_DR[MAIN_DR.n_obs == 20000], "est_headline", "est_crossfit"), "dr_fit",
      {"nosplit": "No-split", "crossfit": "Cross-fitted"}, order=["nosplit", "crossfit"],
      first_column="Nuisance fit", title="Nuisance fitting   (n_obs = 20000, M = 2000)")
sweep(stage(NOSPLIT, "est_init", "est_headline"), "init",
      {"observed": "Observed (P_Y)", "moment": "Moment-matched", "normal": "Standard normal"},
      order=["observed", "moment", "normal"], first_column="Initialisation",
      title="Initialisation   (n_obs = 20000, M = 2000)")
sweep(stage(NOSPLIT, "est_wsweep"), "w_conf",
      {w: str(int(w)) for w in sorted(NOSPLIT.w_conf.unique())}, first_column="w",
      metrics=[("SW2 (no flow)", "sw2_initial"), ("Color-W2 (no flow)", "colour_w2_initial"),
               ("SW2", "sw2_flow"), ("SW2 / floor", "sw2_ratio"), ("ESS on arm", "ess_on_arm")],
      digits=(5, 5, 5, 2, 0),
      title="Confounding strength   (n_obs = 20000, M = 2000, no-split)",
      note="ESS on arm = Kish ESS of the DR weights on the intervened arm, out of n_obs.")

Observations   (M = 2000, no-split)

n_obs               SW2                  SW2 / floor          FID                  Latent SW2           Latent / floor
----------------------------------------------------------------------------------------------------------------------
5000                0.01056 ± 0.00113    1.68 ± 0.25          28.322 ± 1.341       0.09768 ± 0.00534    1.42 ± 0.18
10000               0.01054 ± 0.00086    1.66 ± 0.21          30.326 ± 0.783       0.09108 ± 0.00199    1.29 ± 0.12
15000               0.01066 ± 0.00088    1.67 ± 0.14          30.725 ± 0.737       0.08115 ± 0.00105    1.15 ± 0.05
20000               0.00930 ± 0.00030    1.46 ± 0.04          30.644 ± 0.809       0.08004 ± 0.00114    1.12 ± 0.06

Particles   (n_obs = 20000, no-split)

M (particles)       SW2                  SW2 / floor          FID                  Latent SW2           Latent / floor
------------------------------------------------------------------------------------------------------

Nuisance fitting   (n_obs = 20000, M = 2000)

Nuisance fit        SW2                  SW2 / floor          FID                  Latent SW2           Latent / floor
----------------------------------------------------------------------------------------------------------------------
No-split            0.00930 ± 0.00030    1.46 ± 0.04          30.644 ± 0.809       0.08004 ± 0.00114    1.12 ± 0.06
Cross-fitted        0.00946 ± 0.00018    1.48 ± 0.05          30.785 ± 0.911       0.08075 ± 0.00152    1.13 ± 0.07



Initialisation   (n_obs = 20000, M = 2000)

Initialisation      SW2                  SW2 / floor          FID                  Latent SW2           Latent / floor
----------------------------------------------------------------------------------------------------------------------
Observed (P_Y)      0.00930 ± 0.00030    1.46 ± 0.04          30.644 ± 0.809       0.08004 ± 0.00114    1.12 ± 0.06
Moment-matched      0.01703 ± 0.00057    2.69 ± 0.06          50.020 ± 0.726       0.18701 ± 0.01433    2.65 ± 0.37
Standard normal     0.02173 ± 0.00093    3.43 ± 0.04          63.884 ± 0.934       0.16585 ± 0.00621    2.35 ± 0.23



Confounding strength   (n_obs = 20000, M = 2000, no-split)

w                   SW2 (no flow)        Color-W2 (no flow)   SW2                  SW2 / floor          ESS on arm
------------------------------------------------------------------------------------------------------------------
1                   0.01089 ± 0.00053    0.04758 ± 0.00881    0.00760 ± 0.00026    1.21 ± 0.04          9801 ± 29
2                   0.01769 ± 0.00030    0.09517 ± 0.00085    0.00764 ± 0.00027    1.21 ± 0.07          9208 ± 50
3                   0.02261 ± 0.00096    0.12578 ± 0.00403    0.00820 ± 0.00077    1.29 ± 0.07          8283 ± 62
4                   0.02743 ± 0.00082    0.15413 ± 0.00510    0.00809 ± 0.00042    1.28 ± 0.10          7159 ± 73
5                   0.03162 ± 0.00042    0.18119 ± 0.00270    0.00830 ± 0.00051    1.31 ± 0.03          5854 ± 150
6                   0.03428 ± 0.00058    0.19755 ± 0.00759    0.00844 ± 0.00034    1.33 ± 0.07          4620 ± 144
7                   0.03

**Observations.** The SW2 ratio improves 1.68× → 1.46× over a 4× increase in data, nearly all
of it in the last step; the latent ratio falls steadily 1.42× → 1.12×. FID drifts the other way,
28.322 → 30.644.

**Particles.** The raw column falls 0.01735 → 0.00930 while the ratio *rises* 1.06× → 1.46×, and
the latent ratio is below 1 up to `M = 1000` (0.97 / 0.90 / 0.96). Read the ratio.

**Nuisance fitting.** No difference: 0.00946 ± 0.00018 against 0.00930 ± 0.00030, 1.48× against
1.46×.

**Initialisation.** The largest effect here — SW2 0.00930 → 0.01703 → 0.02173 and FID
30.644 → 50.020 → 63.884 from observed to moment-matched to standard normal, with the latent
column ranking the two Gaussian starts the other way round (0.18701 against 0.16585).

**Confounding.** Over `w = 1 → 10` the logged arm degrades 0.01089 → 0.04332 in SW2 and the
weight ESS collapses 9801 → 1383, while the flow's SW2 moves only 0.00760 → 0.00946
(1.21× → 1.48×).

## 6. Target-weight solver

In [7]:
SOLVER = at.pin_estimator(WITH_EXACT[WITH_EXACT.tag.astype(str).str.startswith("est_solver_")])
rows = [(f"{int(n):<8}{label}",
         [at.summarise(SOLVER[(SOLVER.n_obs == n) & (SOLVER.solver == key)], column)
          for _, column in at.METRICS["ColorMNIST"]])
        for n in sorted(SOLVER.n_obs.unique())
        for key, label in [("projected_nystrom", "projected"), ("exact", "exact")]]
print(at.plain_table(rows, [label for label, _ in at.METRICS["ColorMNIST"]],
                     first_column="n_obs   solver", label_width=20,
                     title=f"ColorMNIST   projected Nystrom against the exact ridge   "
                           f"(M = 2000, DR, seeds {SEEDS}, arms averaged)",
                     note="Both arms ran on the same node in the same submission batch; "
                          "n_obs = 15000 was not run in this stage."))

ColorMNIST   projected Nystrom against the exact ridge   (M = 2000, DR, seeds [0, 1, 2], arms averaged)

n_obs   solver      SW2 ↓                FID ↓                Latent SW2 ↓         Color-W2 ↓
---------------------------------------------------------------------------------------------
5000    projected   0.01057 ± 0.00103    28.384 ± 1.220       0.09758 ± 0.00559    0.02336 ± 0.00602
5000    exact       0.01069 ± 0.00128    28.401 ± 1.264       0.09771 ± 0.00538    0.02461 ± 0.00656
10000   projected   0.01029 ± 0.00061    30.181 ± 1.105       0.09159 ± 0.00245    0.02559 ± 0.00621
10000   exact       0.01019 ± 0.00037    29.947 ± 0.839       0.09105 ± 0.00294    0.02326 ± 0.00373
20000   projected   0.00935 ± 0.00041    30.693 ± 0.841       0.08006 ± 0.00112    0.02075 ± 0.00393
20000   exact       0.00952 ± 0.00037    30.719 ± 0.876       0.08031 ± 0.00105    0.02080 ± 0.00356

Both arms ran on the same node in the same submission batch; n_obs = 15000 was not run in this stage

The approximation costs nothing measurable in sample quality: SW2 reads 0.01057 projected
against 0.01069 exact at `n_obs = 5000`, 0.01029 against 0.01019 at 10000 and 0.00935 against
0.00952 at 20000 — no consistent winner, every pair inside the seed spread of either arm. FID,
latent SW2 and Color-W2 behave the same way.

## 7. Coverage

In [8]:
rows = []
for tag, g in WITH_EXACT.groupby(WITH_EXACT.tag.astype(str)):
    estimators = g.estimator.nunique()
    rows.append((tag, [(len(g), 0.0), (estimators, 0.0),
                       (len(g) / (2 * estimators), 0.0)]))
print(at.plain_table(rows, ["rows", "estimators", "runs"], first_column="Launcher stage",
                     label_width=30, cell_width=13, digits=0, arrow=False,
                     title="ColorMNIST   what is on disk",
                     note="runs = rows / (2 arms x estimators), one per (design cell, data "
                          f"seed); {len(SEEDS)} seeds, 1 repeat each."))
print(f"\ntotal {len(WITH_EXACT)} rows: "
      f"{int((WITH_EXACT.solver == 'projected_nystrom').sum())} projected-Nystrom "
      f"and {int((WITH_EXACT.solver == 'exact').sum())} exact.")

ColorMNIST   what is on disk

Launcher stage                rows         estimators   runs
------------------------------------------------------------
est_crossfit                  72 ± 0       3 ± 0        12 ± 0
est_headline                  18 ± 0       3 ± 0        3 ± 0
est_init                      36 ± 0       3 ± 0        6 ± 0
est_nobs                      54 ± 0       3 ± 0        9 ± 0
est_particles                 54 ± 0       3 ± 0        9 ± 0
est_solver_exact              18 ± 0       1 ± 0        9 ± 0
est_solver_projected_nystrom  18 ± 0       1 ± 0        9 ± 0
est_wsweep                    180 ± 0      3 ± 0        30 ± 0

runs = rows / (2 arms x estimators), one per (design cell, data seed); 3 seeds, 1 repeat each.

total 450 rows: 432 projected-Nystrom and 18 exact.
